# TEST INFERENCE PIPELINE

In [ ]:
from pathlib import Path
import numpy as np
import os
import io
import cv2
from PIL import Image
import torch

from torch.utils.data import DataLoader
from emma import EmmaDataset, EmmaSearcher, SearchQuery
from emma.dataloader import EmmaMapDataset
from emma.dataloader import emma_collate
from transformers import Sam3Config, Sam3Processor, Sam3Model

import utils.logging_ as logging

import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
# Check CUDA Availability

if torch.cuda.is_available():
    print("Ready")
    device = torch.device('cuda')
else:
    print("Not Ready")
    device = torch.device('cpu')

## > DATASET

In [ ]:
dataset_root = None  # e.g. Path("/path/to/EMMa")

if dataset_root is None:
    raise ValueError("Please set `dataset_root` to the path of the EMMa dataset.")

dataset_root = Path(dataset_root)

if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset path does not exist: {dataset_root}")

In [ ]:
# INIT DATASET
ds = EmmaDataset(dataset_root)

In [ ]:
# QUERY
se = EmmaSearcher(ds)
query = SearchQuery(splits=["Test"], require_listed=["mass", "category"], require_present=["package_dimensions"], quantity_range=[1,1], limit=10)
asins = se.query_asins(query)
print(len(asins))

In [ ]:
def my_transform(asin: str, rec: dict):
    image_dict = ds.get_images(rec)
    _, first_img_bytes = next(iter(image_dict.items()))
    first_img = Image.open(io.BytesIO(first_img_bytes)).convert("RGB")
    first_img = np.asarray(first_img)

    title = ds.get_title(rec=rec)

    categories = ds.get_category_names(rec=rec)

    return {
        "image": first_img,
        "title": title,
        "categories": categories,
        "asin": asin,
    }

In [ ]:
batch_size = 4  # ADJUST ACCORDING TO AVAILABLE VRAM

loader = DataLoader(
    EmmaMapDataset(ds, asins, transform=my_transform),
    batch_size=batch_size, shuffle=False, collate_fn=emma_collate,
)

## > MODEL

In [ ]:
sam3_root = None  # e.g. Path("/path/to/SAM3")

if sam3_root is None:
    raise ValueError("Please set `sam3_root` to the path of the SAM3 model.")

sam3_root = Path(sam3_root)

if not sam3_root.exists():
    raise FileNotFoundError(f"Model path does not exist: {sam3_root}")

In [ ]:
RES = 560  # try 560, 700, 840, 1008

config = Sam3Config.from_pretrained(sam3_root)
config.image_size = RES

model_kwargs = {
    "config": config,
    "use_safetensors": True,
}
if device.type == "cuda":
    model_kwargs["device_map"] = "cuda"
    model_kwargs["torch_dtype"] = torch.float16
else:
    model_kwargs["torch_dtype"] = torch.float32

segmentation_model = Sam3Model.from_pretrained(sam3_root, **model_kwargs).eval()
if device.type != "cuda":
    segmentation_model = segmentation_model.to(device)

segmentation_processor = Sam3Processor.from_pretrained(
    sam3_root,
    size={"height": RES, "width": RES},
)
print(f"Model loaded on {device}")

## > INFERENCE

In [ ]:
SEGMENTATION_THRESHOLD = 0.25
MASK_THRESHOLD = 0.5

THRESHOLDING_THRESHOLD = 253
THRESHOLDING_KERNEL_SIZE = 3
THRESHOLDING_OPEN_ITERATIONS = 2

In [ ]:
def most_specific_category(categories):
    if not categories:
        return ""

    category = categories[-1]
    while isinstance(category, (list, tuple)):
        if not category:
            return ""
        category = category[-1]

    return str(category).strip()


def title_segmentation_prompt(images, titles, categories_batch):
    # Process image batch only
    segmentation_inputs = segmentation_processor(
        images=images,
        return_tensors="pt",
    )

    # Tokenize title, forcing truncation to 32 tokens
    text_inputs = segmentation_processor.tokenizer(
        titles,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=32,
    )

    segmentation_inputs.update(text_inputs)
    return segmentation_inputs.to(device)


def category_segmentation_prompt(images, titles, categories_batch):
    prompts = [most_specific_category(categories) for categories in categories_batch]
    return segmentation_processor(
        images=images,
        text=prompts,
        return_tensors="pt",
    ).to(device)


SEGMENTATION_PROMPT_PIPELINE = [
    {
        "name": "truncated title",
        "prompt_fn": title_segmentation_prompt,
    },
    {
        "name": "last category",
        "prompt_fn": category_segmentation_prompt,
    },
]


def run_segmentation_prompt(images, titles, categories_batch, prompt_fn):
    segmentation_inputs = prompt_fn(images, titles, categories_batch)

    with torch.inference_mode():
        segmentation_outputs = segmentation_model(**segmentation_inputs)

    return segmentation_processor.post_process_instance_segmentation(
        segmentation_outputs,
        threshold=SEGMENTATION_THRESHOLD,
        mask_threshold=MASK_THRESHOLD,
        target_sizes=segmentation_inputs["original_sizes"].tolist(),
    )


def thresholding_segmentation(
    image_rgb: np.ndarray,
    threshold: int,
    kernel_size: int,
    open_iterations: int,
) -> np.ndarray:
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    _, bin_img = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY_INV)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    bin_img = cv2.morphologyEx(
        bin_img,
        cv2.MORPH_OPEN,
        kernel,
        iterations=open_iterations,
    )
    return (bin_img > 127).astype(np.uint8, copy=False)


def image_to_rgb_array(image):
    if hasattr(image, "convert"):
        image = np.array(image.convert("RGB"))
    elif torch.is_tensor(image):
        image = image.detach().cpu().numpy()
    else:
        image = np.array(image)

    if image.ndim == 3 and image.shape[0] in (1, 3) and image.shape[-1] not in (1, 3):
        image = np.moveaxis(image, 0, -1)

    if image.dtype != np.uint8:
        if image.max(initial=0) <= 1.0:
            image = image * 255
        image = np.clip(image, 0, 255).astype(np.uint8)

    return image


def thresholding_fallback_mask(image):
    return thresholding_segmentation(
        image_to_rgb_array(image),
        threshold=THRESHOLDING_THRESHOLD,
        kernel_size=THRESHOLDING_KERNEL_SIZE,
        open_iterations=THRESHOLDING_OPEN_ITERATIONS,
    )

In [ ]:
inference_results = []

def as_batch_list(values):
    if isinstance(values, list):
        return values
    if isinstance(values, tuple):
        return list(values)
    return [values]


def get_batch_item(values, index):
    return values[index]


def select_batch_items(values, indices):
    if torch.is_tensor(values) or isinstance(values, np.ndarray):
        return values[indices]
    return [values[i] for i in indices]


def has_detection(segmentation_result):
    scores = segmentation_result["scores"]
    if hasattr(scores, "detach"):
        scores = scores.detach().cpu().numpy()

    return len(scores) > 0 and np.any(np.asarray(scores) >= SEGMENTATION_THRESHOLD)


def segmentation_result_to_mask(image, result):
    image = image_to_rgb_array(image)

    masks = result["masks"]
    if hasattr(masks, "detach"):
        masks = masks.detach().cpu().numpy()

    if masks.ndim == 2:
        masks = masks[None, ...]
    elif masks.ndim == 4 and masks.shape[1] == 1:
        masks = masks[:, 0]

    mask = np.any(masks > MASK_THRESHOLD, axis=0) if len(masks) > 0 else np.zeros(image.shape[:2], dtype=bool)
    return mask.astype(bool)


for batch_idx, batch in enumerate(loader):
    images = batch["image"]
    titles = as_batch_list(batch["title"])
    categories_batch = as_batch_list(batch["categories"])

    batch_results = []
    for i, title in enumerate(titles):
        image = get_batch_item(images, i)
        batch_results.append({
            "image": image,
            "title": title,
            "categories": categories_batch[i],
            "prompt_name": "thresholding threshold",
            "segmentation_result": None,
            "finalized_mask": None,
        })

    unresolved = list(range(len(batch_results)))
    for prompt in SEGMENTATION_PROMPT_PIPELINE:
        if not unresolved:
            break

        prompt_images = select_batch_items(images, unresolved)
        prompt_titles = [titles[i] for i in unresolved]
        prompt_categories = [categories_batch[i] for i in unresolved]

        candidate_results = run_segmentation_prompt(
            prompt_images,
            prompt_titles,
            prompt_categories,
            prompt["prompt_fn"],
        )

        next_unresolved = []
        for sample_idx, candidate_result in zip(unresolved, candidate_results):
            if not has_detection(candidate_result):
                next_unresolved.append(sample_idx)
                continue

            image = get_batch_item(images, sample_idx)
            batch_results[sample_idx]["prompt_name"] = prompt["name"]
            batch_results[sample_idx]["segmentation_result"] = candidate_result
            batch_results[sample_idx]["finalized_mask"] = segmentation_result_to_mask(
                image,
                candidate_result,
            )

        unresolved = next_unresolved

    for sample_idx in unresolved:
        image = get_batch_item(images, sample_idx)
        batch_results[sample_idx]["finalized_mask"] = thresholding_fallback_mask(image)

    inference_results.extend(batch_results)
    logging.progress_bar(batch_idx, len(loader), 1, 1)

print("Done")

## > VISUALIZE

In [ ]:
def show_segmentation_results(image, result):
    image = image_to_rgb_array(image)

    masks = result["masks"]
    boxes = result["boxes"]
    scores = result["scores"]

    if hasattr(masks, "detach"):
        masks = masks.detach().cpu().numpy()
    if hasattr(boxes, "detach"):
        boxes = boxes.detach().cpu().numpy()
    if hasattr(scores, "detach"):
        scores = scores.detach().cpu().numpy()

    if masks.ndim == 2:
        masks = masks[None, ...]
    elif masks.ndim == 4 and masks.shape[1] == 1:
        masks = masks[:, 0]

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(image)

    cmap = plt.get_cmap("tab10")

    for i, (mask, box, score) in enumerate(zip(masks, boxes, scores)):
        color = np.array(cmap(i % 10)[:3])

        overlay = np.zeros((*mask.shape, 4))
        overlay[..., :3] = color
        overlay[..., 3] = (mask > MASK_THRESHOLD) * 0.4
        ax.imshow(overlay)

        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)

        ax.text(x1, y1, f"{score:.2f}", color="white")

    ax.axis("off")
    plt.show()


def show_final_mask(image, mask):
    image = image_to_rgb_array(image)

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(image)

    overlay = np.zeros((*mask.shape, 4))
    overlay[..., 1] = 1.0
    overlay[..., 3] = mask * 0.4
    ax.imshow(overlay)

    ax.axis("off")
    plt.show()

In [ ]:
for result in inference_results[:10]:
    print(result["title"])
    print(result["categories"])
    print(f"Selected segmentation fallback: {result['prompt_name']}")

    if result["segmentation_result"] is not None:
        show_segmentation_results(result["image"], result["segmentation_result"])
    else:
        print("Title/category did not detect above threshold; using thresholding fallback.")

In [ ]:
for result in inference_results[:10]:
    print(result["title"])
    print(f"Final mask from: {result['prompt_name']}")
    show_final_mask(result["image"], result["finalized_mask"])